# Como um computador funciona

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Dizer o que fazem a CPU, a RAM e o armazenamento, e por que um programa é copiado para a RAM para rodar
- Acompanhar o ciclo de busca, decodificação e execução de um programa curto
- Explicar por que os caches deixam os programas mais rápidos, e escolher o padrão de acesso que os aproveita
- Descrever como o javac e a JVM transformam código Java em instruções que a CPU executa

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check(...) imprime ✓ ou ✗; os exercícios abaixo usam
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": esperado " + expected + ", veio " + actual);
    }
}

## Em resumo

Um computador tem quatro partes principais. A **CPU** executa instruções; a **RAM** guarda os programas e dados em uso agora, e esquece tudo quando a energia é desligada; o **armazenamento** (um SSD ou um disco) guarda os arquivos; os dispositivos de **entrada e saída** conversam com o mundo. Para rodar, um programa é copiado do armazenamento para a RAM, porque o armazenamento é lento demais para a CPU ler instruções dele.

A CPU repete um ciclo bilhões de vezes por segundo: ela **busca** a instrução cujo endereço está no **contador de programa**, **decodifica** o que ela pede, **executa** e passa para a próxima, a menos que a instrução seja um **salto**, que é como `if` e laços funcionam nesse nível. Um clock de 3 GHz bate 3 bilhões de vezes por segundo.

![O ciclo de busca, decodificação e execução](cycle.pt-br.svg)

#### 🎞️ O ciclo de busca, decodificação e execução

![Um programa de três instruções rodado passo a passo: o contador de programa avança e o acumulador vai de 0 a 5 e a 8](cpu-cycle-steps.pt-br.svg)

### ✏️ Exercício: Uma CPU de três instruções

Um acumulador começa em 0. Rode cada instrução em ordem: `ADD n`, `SUB n` ou `MUL n`, e devolva o acumulador no fim. Esse laço é o ciclo de busca, decodificação e execução.

In [ ]:
int run(String[] program) {
    // TODO: separe cada instrução em nome e número, e aplique-a
    return 0;
}

void main() {
    check("soma e multiplica", run(new String[] {"ADD 5", "MUL 3"}), 15);
    check("subtrai", run(new String[] {"ADD 10", "SUB 4", "SUB 7"}), -1);
    check("sem programa", run(new String[] {}), 0);
}

<details>
<summary>Ver uma resposta</summary>

```java
int run(String[] program) {
    int acc = 0;
    for (String instruction : program) {
        String[] parts = instruction.split(" ");
        int n = Integer.parseInt(parts[1]);
        switch (parts[0]) {
            case "ADD" -> acc += n;
            case "SUB" -> acc -= n;
            case "MUL" -> acc *= n;
            default -> throw new IllegalArgumentException(instruction);
        }
    }
    return acc;
}

void main() {
    check("soma e multiplica", run(new String[] {"ADD 5", "MUL 3"}), 15);
    check("subtrai", run(new String[] {"ADD 10", "SUB 4", "SUB 7"}), -1);
    check("sem programa", run(new String[] {}), 0);
}
```

</details>

A RAM é cerca de 100 vezes mais lenta que a CPU, então os processadores guardam os dados usados recentemente em **caches** pequenos e rápidos. Um dado que já está no cache (um **acerto**) chega em cerca de 1 ns; uma **falha** espera cerca de 100 ns pela RAM. Uma falha traz um bloco inteiro de bytes vizinhos, então ler um array em ordem é muito mais rápido do que pular de um lugar para outro nele.

**Experimente:** os mesmos 16 milhões de números, somados linha por linha e depois coluna por coluna. Linha por linha lê a memória em ordem, então a maioria das leituras acerta o cache.

In [ ]:
int n = 4000;
int[][] grade = new int[n][n];
for (int[] linha : grade) {
    Arrays.fill(linha, 1);
}

long inicio = System.nanoTime();
long porLinhas = 0;
for (int l = 0; l < n; l++) {
    for (int c = 0; c < n; c++) {
        porLinhas += grade[l][c];
    }
}
long meio = System.nanoTime();
long porColunas = 0;
for (int c = 0; c < n; c++) {
    for (int l = 0; l < n; l++) {
        porColunas += grade[l][c];
    }
}
long fim = System.nanoTime();

IO.println("linha por linha:   " + (meio - inicio) / 1_000_000 + " ms (soma " + porLinhas + ")");
IO.println("coluna por coluna: " + (fim - meio) / 1_000_000 + " ms (soma " + porColunas + ")");

### ✏️ Exercício: Mesmo bloco de cache?

Um cache move a memória em blocos: com blocos de 64 bytes, os endereços 0 a 63 são o bloco 0, 64 a 127 o bloco 1, e assim por diante. Dois endereços caem no mesmo bloco?

In [ ]:
boolean sameBlock(int a, int b, int blockSize) {
    // TODO: compare os números de bloco de a e b
    return false;
}

void main() {
    check("0 e 63", sameBlock(0, 63, 64), true);
    check("63 e 64", sameBlock(63, 64, 64), false);
    check("130 e 140", sameBlock(130, 140, 16), true);
}

<details>
<summary>Ver uma resposta</summary>

```java
boolean sameBlock(int a, int b, int blockSize) {
    return a / blockSize == b / blockSize;
}

void main() {
    check("0 e 63", sameBlock(0, 63, 64), true);
    check("63 e 64", sameBlock(63, 64, 64), false);
    check("130 e 140", sameBlock(130, 140, 16), true);
}
```

</details>

O Java acrescenta um passo: o `javac` compila o seu código para **bytecode**, e a **JVM** o executa, transformando o código que mais roda em código de máquina de verdade para a sua CPU.

**O seu computador, do jeito que o Java o vê:**

In [ ]:
Runtime runtime = Runtime.getRuntime();
IO.println("Versão do Java:       " + System.getProperty("java.version"));
IO.println("Sistema operacional:  " + System.getProperty("os.name") + " (" + System.getProperty("os.arch") + ")");
IO.println("Núcleos do processador: " + runtime.availableProcessors());
IO.println("Memória que a JVM pode usar: " + runtime.maxMemory() / (1024 * 1024) + " MiB");

## Leituras

- [The Central Processing Unit (CPU)](https://www.youtube.com/watch?v=FZGugFqdr60) (Crash Course Computer Science #7, 10 min): Acompanhe uma instrução pela busca, decodificação e execução. Repare no registrador de endereço da instrução (o contador de programa) e em como o clock conduz cada passo.
- [Advanced CPU Designs](https://www.youtube.com/watch?v=rtAlC5J1U40) (Crash Course Computer Science #9, 12 min): Por que a RAM é lenta perto da CPU, como um cache guarda blocos inteiros de dados por perto, e o que são um acerto e uma falha de cache. Pipeline e múltiplos núcleos são extras.
- [How JVM Works: JVM Architecture](https://www.geeksforgeeks.org/java/how-jvm-works-jvm-architecture/) (GeeksforGeeks, 8 min): O que o javac produz, como o carregador de classes traz os arquivos .class para a memória, e como o compilador JIT transforma em código de máquina o código que roda muito.

## Verifique

1. O que o contador de programa guarda, e quando ele não passa simplesmente para a próxima instrução?
2. Por que ler um array grande em ordem pode ser dez vezes mais rápido do que lê-lo em ordem aleatória?
3. O que o `javac` produz, e o que transforma isso em instruções que a sua CPU consegue executar?

O quiz confere essas respostas, e os desafios deixam você construir uma CPU pequena, uma máquina de pilha e um cache.

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Decodificador de instruções** · Fácil
- **Máquina de pilha** · Médio
- **Simulador de cache** · Médio
- **CPU minúscula** · Difícil